In [ ]:
import sys
print (sys.version)

In [ ]:
%pip install -q llama-index-core llama-index-readers-file llama-index-llms-google-genai llama-index-embeddings-huggingface pypdf python-dotenv nest-asyncio


In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import dotenv
import nest_asyncio
import pypdf

print("packages imported successfully ")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Gemini API key loaded successfully ")
else:
    print("key not found. check your .env file and its location")

In [ ]:
#llamaindex provides a googlegenai integration for gemini 

import nest_asyncio
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI

nest_asyncio.apply()
Settings.llm = GoogleGenAI(
    model = "gemini-flash-latest",
    api_key = api_key,
    temperature=0,
)

print("gemini configured ")

In [ ]:
response = Settings.llm.complete(
    "replu with only: gemini connection successful"
)

print(response.text)

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("Local embedding model loaded.")

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

pdf_path = Path("document.pdf")
assert pdf_path.exists(), "place document.pdf beside your notebook"

documents = SimpleDirectoryReader(
    input_files=[str(pdf_path)]
).load_data()

print(f"Loaded {len(documents)} document objects")
print(documents[0].text[:500])

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size = 1024)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} nodes.")

In [ ]:
print("first node metadata:")
print(nodes[0].metadata)

print("\nFirst node text:")
print(nodes[0].get_content()[:800])

In [ ]:
page_labels = list(dict.fromkeys(
    str(node.metadata.get("page_label"))
    for node in nodes
))

print("Available page labels:", page_labels)

In [ ]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

print("summary and vector indexes created")

In [ ]:
#create the query engine 

summary_query_engine = summary_index.as_query_engine(
    response_mode = "tree_summarize",
    use_async = True,
)

vector_query_engine = vector_index.as_query_engine(
    similarity_top_k = 2,
)

print("both query engines are ready ")

In [ ]:
from llama_index.core.tools import QueryEngineTool

summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "useful for summarizing the entire knowledge intensive agents"
        "article or giving a broad overview of its main themes "
    ),


)

vector_tool = QueryEngineTool.from_defaults(
    name="vector_tool",
    query_engine=vector_query_engine,
    description=(
        "useful for answeting specific questions about concepts"
        "methods, and findings in the knowledge intensive agents article"
    ),
)

print("both tools created")

In [ ]:
#creting the router 


from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMMultiSelector

router_query_engine = RouterQueryEngine(
    selector=LLMMultiSelector.from_defaults(llm=Settings.llm),
    query_engine_tools=[summary_tool, vector_tool],
    verbose=True
)

print("router ready ")

In [ ]:
response = router_query_engine.query(
    "how does retrieval augmented generation help knowledge intensive agents ?"
)

print(str(response))

print("\nSource page labels")
for source in response.source_nodes:
    print(source.node.metadata.get("page_labels"))

In [ ]:
for i, source in enumerate(response.source_nodes, start=1):
    print(f"Source {i}")
    print("Node ID:", source.node.node_id)
    print("Metadata:", source.node.metadata)
    print("Text preview:", source.node.get_content()[:400])
    print()

In [ ]:
original_nodes = {node.node_id: node for node in nodes}

for source in response.source_nodes:
    original = original_nodes.get(source.node.node_id)

    if original is not None:
        print("Original metadata:", original.metadata)
    else:
        print("Returned node ID was not found in the original nodes.")

In [ ]:
for source in response.source_nodes:
    print("Page label:", source.node.metadata.get("page_label"))

In [ ]:
for i, source in enumerate(response.source_nodes, start=1):
    print(f"\nSOURCE {i} -PAGE {source.node.metadata['page_label']}")
    print(source.node.get_content())

In [ ]:
summary_response = router_query_engine.query(
    "give a concise overview of the entire article, including its main argument, agent roles, proposed framework and future directions"
)

print(str(summary_response))

In [ ]:
#defining function - which builds retrieval filters from its inputs
#wrap is using FunctionTool - lets the llm call that function 
#give it a page specific qn- lets the llm supply arguments such as pg no
#inspect the returned sources - checks whether the pahe restriction worked 


#above - llm chose an engine 
#now i will make it choose a tool and supplu retriebal parameters 

In [ ]:
#define the retrival function 

from typing import List 
from llama_index.core.vector_stores import MetadataFilters, FilterCondition, MetadataFilter

def vector_query(query: str, page_numbers:List[str]):
    """Answer a specific question about the article using vector retrieval.
    
    query: the question to search for.
    page_numbers: pdf page labels to search, from "18" through "44". 
    use an empty list [] to search all pages. 
    
    """

    invalid_pages = set(page_numbers) - set(page_labels)
    if invalid_pages:
        raise ValueError(f"Unknown page labels: {sorted(invalid_pages)}")

    filters = None

    if page_numbers:
        filters = MetadataFilters(
            filters = [
                MetadataFilter(key="page_label", value = page)
                for page in page_numbers
            ],
            condition=FilterCondition.OR
        )

    engine = vector_index.as_query_engine(
        similarity_top_k = 2,
        filters = filters,
    )

    return engine.query(query)

In [ ]:
#wrapping the function as a tool to be called by an llm 

from llama_index.core.tools import FunctionTool

filtered_vector_tool = FunctionTool.from_defaults(
    name = "page_filtered_vector_tool",
    fn = vector_query,
)

print("page-filtered retriecal tool ready ")


# functiontool- wraps any python function that u feed into pass the tools to an llm you have to import 
# the llm module and then call predict and cal function 

# it takes a set of tools as well as an inpit prompt string or a series of chat messages 
# and ihen it able to oth make a decision of the tool to call and call the tool itself and get back the final response


In [ ]:
#gemini selects and call the tool 

tool_response = Settings.llm.predict_and_call(
    [filtered_vector_tool, summary_tool],
    (
        "according to page label 19 only, what roles do knowledge"
        "acquisition and knowledge utilization agents perform?"
    ),
    verbose = True,
)

print(str(tool_response))

In [ ]:
page_19_filters = MetadataFilters(
    filters=[
        MetadataFilter(key="page_label", value="19")
    ],
)

retriever = vector_index.as_retriever(
    similarity_top_k=2,
    filters=page_19_filters,
)

retrieved_nodes = retriever.retrieve(
    "knowledge acquisition and knowledge utilization agents roles"
)

print("Retrieved chunks:", len(retrieved_nodes))

for source in retrieved_nodes:
    print("Metadata:", source.node.metadata)
    print("Text preview:", source.node.get_content()[:300])
    print()

In [ ]:
or_filters = MetadataFilters(
    filters=[MetadataFilter(key="page_label", value="19")],
    condition=FilterCondition.OR,
)

or_nodes = vector_index.as_retriever(
    similarity_top_k=2,
    filters=or_filters,
).retrieve("knowledge acquisition and knowledge utilization agents roles")

print("Chunks retrieved with OR:", len(or_nodes))

In [ ]:
direct_response = vector_query(
    query="What roles do knowledge acquisition and utilization agents perform?",
    page_numbers=["19"],
)

print("Answer:", str(direct_response))
print("Source chunks:", len(direct_response.source_nodes))

In [ ]:
test_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=or_filters,
    llm=Settings.llm,
)

test_response = test_engine.query(
    "What roles do knowledge acquisition and utilization agents perform?"
)

print("Answer:", str(test_response))
print("Source chunks:", len(test_response.source_nodes))

In [ ]:
print("Source chunks:", len(tool_response.source_nodes))

for source in tool_response.source_nodes:
    print("Page label:", source.node.metadata.get("page_label"))
    print(source.node.get_content())
    print()